In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 19:50:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
spark.sparkContext.uiWebUrl

'http://mac.lan:4040'

In [5]:
!wget -P ./data/ https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-04 19:50:17--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-05T02%3A43%3A14Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-05T01%3A43%3A11Z&ske=2026-03-05T02%3A43%3A14Z&sks=b&skv=2018-11-09&sig=9WUw4pVw60pY6E1rYPOr8hxXaQr2sZkNxqpWmSmXHhE%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjY3OTAxNywibmJmIjoxNzcyNjc1NDE3LCJwYXRoIjoi

In [6]:
df_inferred = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("./data/fhvhv_tripdata_2021-01.csv.gz")

In [7]:
df_inferred.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [8]:
# save schema to json
schema_json = df_inferred.schema.json()
print(schema_json)

{"fields":[{"metadata":{},"name":"hvfhs_license_num","nullable":true,"type":"string"},{"metadata":{},"name":"dispatching_base_num","nullable":true,"type":"string"},{"metadata":{},"name":"pickup_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"dropoff_datetime","nullable":true,"type":"timestamp"},{"metadata":{},"name":"PULocationID","nullable":true,"type":"integer"},{"metadata":{},"name":"DOLocationID","nullable":true,"type":"integer"},{"metadata":{},"name":"SR_Flag","nullable":true,"type":"integer"}],"type":"struct"}


In [9]:
# save schema to file
import json
from pathlib import Path

schema_path = Path("./schemas/fhvhv_tripdata.schema.json")

# pretty-print for readability
schema_dict = json.loads(schema_json)

with schema_path.open("w") as f:
    json.dump(schema_dict, f, indent=2)

print(f"Schema saved to {schema_path.resolve()}")

Schema saved to /Users/mcopple/src/zoomcamps/src/de-homework-2026/module-6/examples/notebooks/schemas/fhvhv_tripdata.schema.json


In [10]:
# now try to read the data again, but this time with the schema
from pyspark.sql.types import StructType
with open("./schemas/fhvhv_tripdata.schema.json") as f:
    schema_dict = json.load(f)

schema = StructType.fromJson(schema_dict)

df = spark.read \
    .option("header", True) \
    .schema(schema) \
    .csv("./data/fhvhv_tripdata_2021-01.csv.gz")

In [11]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', IntegerType(), True)])

In [12]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [13]:
# repartition dataframe to enable parallel processing
df = df.repartition(24)

In [14]:
df.write.parquet('./data/fhvhv/2021/01/')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/Users/mcopple/src/zoomcamps/src/de-homework-2026/module-6/examples/notebooks/data/fhvhv/2021/01 already exists. Set mode as "overwrite" to overwrite the existing path. SQLSTATE: 42K04

In [15]:
df = spark.read.parquet('./data/fhvhv/2021/01/')

In [ ]:
df

In [16]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID').filter(df.hvfhs_license_num == 'HV0003').show(
    truncate=False
)

+-------------------+-------------------+------------+------------+
|pickup_datetime    |dropoff_datetime   |PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-02 19:23:33|2021-01-02 19:31:20|223         |179         |
|2021-01-13 08:58:46|2021-01-13 09:02:49|181         |106         |
|2021-01-29 15:21:13|2021-01-29 15:38:18|226         |82          |
|2021-01-18 21:37:13|2021-01-18 21:48:38|61          |72          |
|2021-01-02 18:50:17|2021-01-02 19:04:44|17          |256         |
|2021-01-24 08:18:27|2021-01-24 08:31:15|196         |173         |
|2021-01-16 19:18:50|2021-01-16 19:27:42|23          |118         |
|2021-01-16 23:05:14|2021-01-16 23:12:57|61          |61          |
|2021-01-31 13:24:26|2021-01-31 13:36:07|249         |170         |
|2021-01-07 16:21:50|2021-01-07 16:33:42|130         |28          |
|2021-01-05 15:23:53|2021-01-05 15:50:56|138         |205         |
|2021-01-18 13:15:20|2021-01-18 13:45:55|141    

In [17]:
from pyspark.sql import functions as F

In [19]:
(df
    .withColumn('pickup_date', F.to_date(df.pickup_datetime))
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime))
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID')
 ).show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-02|  2021-01-02|         223|         179|
| 2021-01-12|  2021-01-12|         197|          86|
| 2021-01-22|  2021-01-22|          76|          63|
| 2021-01-13|  2021-01-13|         181|         106|
| 2021-01-23|  2021-01-23|         116|          42|
| 2021-01-29|  2021-01-29|         226|          82|
| 2021-01-18|  2021-01-18|          61|          72|
| 2021-01-02|  2021-01-02|          17|         256|
| 2021-01-24|  2021-01-24|         196|         173|
| 2021-01-16|  2021-01-16|          23|         118|
| 2021-01-16|  2021-01-16|          61|          61|
| 2021-01-31|  2021-01-31|         249|         170|
| 2021-01-16|  2021-01-16|         234|          87|
| 2021-01-25|  2021-01-25|         210|          29|
| 2021-01-07|  2021-01-07|         130|          28|
| 2021-01-05|  2021-01-05|         138|       

In [20]:


def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    else:
        return f'e/{num:03x}'

In [21]:
crazy_stuff('B02884')

's/b44'

In [22]:
# Turn it into a UDF
crazy_stuff_udf = F.udf(crazy_stuff, returnType=F.StringType())

In [23]:
(df
    .withColumn('pickup_date', F.to_date(df.pickup_datetime))
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime))
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num))
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID')
 ).show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/b3b| 2021-01-02|  2021-01-02|         223|         179|
|  e/9ce| 2021-01-12|  2021-01-12|         197|          86|
|  e/9ce| 2021-01-22|  2021-01-22|          76|          63|
|  e/b37| 2021-01-13|  2021-01-13|         181|         106|
|  e/9ce| 2021-01-23|  2021-01-23|         116|          42|
|  e/a39| 2021-01-29|  2021-01-29|         226|          82|
|  e/a39| 2021-01-18|  2021-01-18|          61|          72|
|  s/acd| 2021-01-02|  2021-01-02|          17|         256|
|  e/b14| 2021-01-24|  2021-01-24|         196|         173|
|  e/acc| 2021-01-16|  2021-01-16|          23|         118|
|  s/acd| 2021-01-16|  2021-01-16|          61|          61|
|  s/b13| 2021-01-31|  2021-01-31|         249|         170|
|  e/9ce| 2021-01-16|  2021-01-16|         234|          87|
|  e/9ce| 2021-01-25|  2